# Agent & vector store — small-token solution

We replace the speech with an original, fictional Python Club handbook and pair it with a tiny Ruff reference. Both files are in `datasets/`. Ruff facts are paraphrased from [official documentation](https://docs.astral.sh/ruff/).

The pipeline follows the lab: load → split → embed → Chroma → RetrievalQA tools → agent. We retain the lab's LangChain 0.2 APIs for compatibility.

Cost controls: cached document and query embeddings; persistent collections; two chunks per retrieval; short answers; at most three agent iterations; no automatic API retries; four test invocations. Re-running the tests still costs tokens. Token reporting below covers chat calls, not embeddings.


In [1]:
# Run once if these packages are not installed, then restart the kernel.
# %pip install -r requirements-solution.txt


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain.agents import AgentType, Tool, initialize_agent
from langchain.chains import RetrievalQA
from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain_community.callbacks.manager import get_openai_callback
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
import hashlib

ROOT = Path.cwd()
assert (ROOT / "datasets/python_club.txt").exists(), "Open this notebook from the project folder."
load_dotenv(ROOT / ".env")
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env before running."
# Avoid sending traces to another service through inherited settings.
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGSMITH_TRACING"] = "false"
MODEL = os.getenv("LAB_MODEL", "gpt-4o-mini")
llm = ChatOpenAI(model=MODEL, temperature=0, max_tokens=180,
                 max_retries=0, timeout=45)
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small", max_retries=0)
embeddings = CacheBackedEmbeddings.from_bytes_store(
    embedding_model, LocalFileStore(str(ROOT / ".lab_cache/embeddings")),
    namespace="text-embedding-3-small", query_embedding_cache=True,
)
print("Configured model:", MODEL)


/Users/mithila/AI-Engineering/week16/labs/lab-agent-vector-store/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Configured model: gpt-4o-mini


In [3]:
splitter = RecursiveCharacterTextSplitter(chunk_size=420, chunk_overlap=40)

def build_store(filename, name):
    docs = TextLoader(str(ROOT / "datasets" / filename), encoding="utf-8").load()
    chunks = splitter.split_documents(docs)
    # Content-based collection name prevents stale results when the data changes.
    digest = hashlib.sha256(("420:40:text-embedding-3-small:" +
                             docs[0].page_content).encode()).hexdigest()[:12]
    db = Chroma(collection_name=f"{name}-{digest}", embedding_function=embeddings,
                persist_directory=str(ROOT / ".lab_cache/chroma"))
    if not db.get()["ids"]:
        db.add_documents(chunks, ids=[f"{name}-{i}" for i in range(len(chunks))])
    print(f"{name}: {len(chunks)} chunks; retrieve at most 2")
    return db

club_db = build_store("python_club.txt", "python-club")
ruff_db = build_store("ruff_notes.txt", "ruff-reference")


/var/folders/dx/f6_nz67d6cv3qrl0cbvdk0ww0000gn/T/ipykernel_90451/124951505.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  db = Chroma(collection_name=f"{name}-{digest}", embedding_function=embeddings,


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


python-club: 2 chunks; retrieve at most 2


ruff-reference: 2 chunks; retrieve at most 2


In [4]:
qa_prompt = PromptTemplate.from_template(
    "Use only the context to answer in at most two short sentences. "
    "If the context does not establish the answer, say so.\n"
    "Context: {context}\nQuestion: {question}\nAnswer:"
)
def make_qa(db):
    return RetrievalQA.from_chain_type(
        llm=llm, chain_type="stuff", retriever=db.as_retriever(search_kwargs={"k": 2}),
        chain_type_kwargs={"prompt": qa_prompt},
    )
club_qa, ruff_qa = make_qa(club_db), make_qa(ruff_db)

def make_agent(return_direct=False):
    tools = [
        Tool(name="Python_Club_QA", func=club_qa.run, return_direct=return_direct,
             description="Answers questions about Python Club meetings, submission rules and current tooling. Ask a complete question."),
        Tool(name="Ruff_QA", func=ruff_qa.run, return_direct=return_direct,
             description="Answers questions about Ruff capabilities, linting, import sorting and replacing other tools. Ask a complete question."),
    ]
    return initialize_agent(
        tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        max_iterations=3, early_stopping_method="force",
        handle_parsing_errors=False, return_intermediate_steps=True, verbose=False,
        agent_kwargs={"prefix": "Answer using the relevant tools. For questions involving both the club and Ruff, consult BOTH tools before answering. Keep tool questions and final answers brief."},
    )
agent = make_agent()
router = make_agent(return_direct=True)


/var/folders/dx/f6_nz67d6cv3qrl0cbvdk0ww0000gn/T/ipykernel_90451/1785395094.py:20: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  return initialize_agent(


## Test routing and multi-hop reasoning

Each normal single-source test typically needs three chat calls (routing, retrieval answer, final answer); multi-hop typically needs five; direct return typically needs two. This is roughly 13 chat calls, not four. The iteration limit bounds extra attempts. We print tool names rather than long agent logs.


In [5]:
results = []
def run_test(label, runner, question, expected_tools):
    with get_openai_callback() as usage:
        result = runner.invoke({"input": question})
    used = [action.tool for action, _ in result["intermediate_steps"]]
    passed = set(used) == set(expected_tools)
    row = {"test": label, "tools": used, "routing_ok": passed,
           "answer": result["output"], "chat_tokens": usage.total_tokens,
           "chat_calls": usage.successful_requests}
    results.append(row)
    print(label, "| tools:", used, "| routing OK:", passed)
    print(result["output"])
    print("Chat tokens:", usage.total_tokens, "| calls:", usage.successful_requests)
    return result

club_question = "When and where does the Python Club meet?"
direct = run_test("Club", agent, club_question, ["Python_Club_QA"])


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Club | tools: ['Python_Club_QA'] | routing OK: True
The Python Club meets on Wednesdays at 18:00 in Room 204.
Chat tokens: 945 | calls: 3


In [6]:
ruff_result = run_test("Ruff", agent,
    "What is Ruff implemented in, and can it fix unused imports?", ["Ruff_QA"])


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Ruff | tools: ['Ruff_QA'] | routing OK: True
Ruff is implemented in Rust and can automatically fix unused imports.
Chat tokens: 1003 | calls: 3


In [7]:
combined = run_test("Both sources", agent,
    "Which tools does the Python Club currently use, and could Ruff replace them while supporting the club's import requirements? Consult both sources.",
    ["Python_Club_QA", "Ruff_QA"])


Both sources | tools: ['Python_Club_QA', 'Ruff_QA'] | routing OK: True
The Python Club currently uses Flake8 for linting and isort for import sorting. Ruff can replace both tools while supporting the club's import requirements.
Chat tokens: 1853 | calls: 5


In [8]:
routed = run_test("Direct return", router, club_question, ["Python_Club_QA"])
assert len(routed["intermediate_steps"]) == 1
assert routed["output"] == routed["intermediate_steps"][0][1]
print("Total chat tokens:", sum(r["chat_tokens"] for r in results))
print("Total chat calls:", sum(r["chat_calls"] for r in results))


Direct return | tools: ['Python_Club_QA'] | routing OK: True
The Python Club meets on Wednesdays at 18:00 in Room 204.
Chat tokens: 537 | calls: 2
Total chat tokens: 4338
Total chat calls: 13


## Reflection

Expected behavior: the meeting question uses Python_Club_QA, the Ruff question uses Ruff_QA, and the combined question uses both. The expected combined answer is that the club uses Flake8 and isort; Ruff offers linting, unused-import fixes, and import sorting, so it can cover these requirements with appropriate configuration.

With `return_direct=True`, the agent returns the first tool's answer immediately. This saves the final synthesis call for a single-source question, but prevents this agent from gathering a second tool result for a multi-hop answer. Without it, the agent can consult both tools and combine their answers.

Actual observations are generated below from the executed tests, rather than assuming routing succeeded. A correct tool choice alone does not prove factual correctness: compare the short answers with the two source files.


In the completed run, all four tests selected the expected tools. The combined answer correctly identified Flake8 and isort and explained that Ruff can replace both. Normal single-source answers took three chat calls each; direct return took two (537 versus 945 tokens for the same club question). The multi-hop answer took five calls. Total: 4,338 chat tokens across 13 calls, excluding embedding usage.

In [9]:
from IPython.display import Markdown, display
observations = "\n".join(
    f"- **{r['test']}**: used {', '.join(r['tools'])}; "
    f"routing {'matched' if r['routing_ok'] else 'did not match'} expectations; "
    f"{r['chat_calls']} chat calls, {r['chat_tokens']} tokens."
    for r in results
)
display(Markdown("### Observed results\n" + observations))
assert all(r["routing_ok"] for r in results), "Inspect unexpected routing before submitting."


### Observed results
- **Club**: used Python_Club_QA; routing matched expectations; 3 chat calls, 945 tokens.
- **Ruff**: used Ruff_QA; routing matched expectations; 3 chat calls, 1003 tokens.
- **Both sources**: used Python_Club_QA, Ruff_QA; routing matched expectations; 5 chat calls, 1853 tokens.
- **Direct return**: used Python_Club_QA; routing matched expectations; 2 chat calls, 537 tokens.